# CS781 Phase 5 — Legal Statute Identification (LSI), multi-label
Every experiment starts from the Phase 1 LegalLM (decoder-only, 12L / 768d, ctx 1024).

Model: document -> overlapping chunks spread over the whole judgment -> Phase 1 backbone ->
chunk vectors [mean ; last-token] -> 1 transformer layer across chunks -> **label-wise attention**
(each statute has its own query that picks the chunks holding its evidence) -> one logit per statute.
Loss: BCE with per-statute positive weights (rare statutes up-weighted), bias initialised to log prior.
Thresholds: global vs per-statute (support-shrunk), chosen by 2-fold cross-fitting on dev; optional
"at least one statute per document" rule kept only if it helps on dev.

| Submission | Technique | Representation |
|---|---|---|
| 01_fullft_lwan | Full fine-tuning (KD teacher) | 4 chunks x 512 spread over the doc |
| 02_lora_lwan | LoRA on qkv/proj/w12/w3 (compulsory) | same |
| 03_adapter_lwan | Bottleneck adapters | same |
| 04_kd_student6 | Knowledge distillation 12L teacher -> 6L student | same |
| 05_partial_top4 | Partial / layer-wise FT (top 4 blocks) | same |
| 06_prompt_tuning | Prompt tuning (20 soft tokens, backbone frozen) | same |
| 07_ensemble_avg | Probability average of 01–06 | — |
| 08_lora_first1024 | LoRA, flat baseline | first 1024 tokens only |

In [ ]:
# Config
import os, sys, json, math, time, random, re, shutil, zipfile, gc, csv
from collections import Counter
from contextlib import nullcontext
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint
from sklearn.metrics import f1_score

SMOKE = os.environ.get("LSI_SMOKE", "0") == "1"   # set True for a tiny dry run
RUN_START = time.time()

INPUT_ROOT = os.environ.get("LSI_INPUT", "/kaggle/input")
OUT = os.environ.get("LSI_OUT", "/kaggle/working")
os.makedirs(OUT, exist_ok=True)

CFG = dict(
    seed=42,
    chunk_len=512, overlap=64,
    max_chunks=4,          # chunks per document
    select="spread",       # 'spread' = evenly spaced over the whole doc | 'first' | 'headtail'
    micro_docs=4, accum=4,  # effective batch = 16 docs
    infer_docs=16,
    kd_pool=8000,          # training docs the KD student uses (teacher logits computed for these)
    n_evals=3,             # dev evaluations per run (best checkpoint kept)
    eval_subset=2000,      # dev docs used for the in-training evaluations (final eval uses full dev)
    pos_weight_max=20.0,   # cap on per-statute positive weight
    total_hours=4.6,       # hard wall-clock guard for the whole notebook
    holdout_frac=0.2,      # only used when train.json is missing
)

EXPS = [
    dict(name="01_fullft_lwan",    method="full",    lr=3e-5, head_lr=1e-3, epochs=4, minutes=40),
    dict(name="02_lora_lwan",      method="lora",    lr=4e-4, head_lr=1e-3, epochs=6, minutes=30, r=16, alpha=32),
    dict(name="03_adapter_lwan",   method="adapter", lr=1e-3, head_lr=1e-3, epochs=6, minutes=30, bottleneck=64),
    dict(name="04_kd_student6",    method="kd",      lr=6e-5, head_lr=1e-3, epochs=6, minutes=25,
         layers=[0, 2, 4, 7, 9, 11], T=2.0, alpha=0.6),
    dict(name="05_partial_top4",   method="partial", lr=6e-5, head_lr=1e-3, epochs=6, minutes=18, train_top=4),
    dict(name="06_prompt_tuning",  method="prompt",  lr=3e-2, head_lr=1e-3, epochs=6, minutes=20, n_prompt=20),
    dict(name="08_lora_first1024", method="lora",    lr=4e-4, head_lr=1e-3, epochs=6, minutes=15, r=16, alpha=32,
         chunk_len=1024, max_chunks=1, select="first", micro_docs=4, accum=4, optional=True),
]

if SMOKE:
    CFG.update(chunk_len=64, overlap=8, max_chunks=3, micro_docs=2, accum=2, infer_docs=4,
               kd_pool=16, n_evals=2, eval_subset=12, total_hours=1.0)
    for e in EXPS:
        e.update(epochs=1, minutes=0.3)
        if e["name"].startswith("08"):
            e.update(chunk_len=128, micro_docs=2, accum=2)
        if e["method"] == "prompt":
            e.update(n_prompt=4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print("device:", device, torch.cuda.get_device_name(0) if USE_AMP else "")


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


seed_all(CFG["seed"])


def elapsed_h():
    return (time.time() - RUN_START) / 3600

In [ ]:
# Locate inputs (searches every attached dataset)
def find_files(name):
    hits = []
    for root, _, files in os.walk(INPUT_ROOT, followlinks=True):
        if name in files:
            hits.append(os.path.join(root, name))
    return sorted(hits)


LABEL_KEYS = ("labels", "label", "statutes", "statute_ids", "statute", "targets")
TEXT_KEYS = ("text", "document", "facts", "content", "doc", "case_text")
ID_KEYS = ("id", "case_id", "doc_id", "ID")


def parse_labels(v):
    """Accepts [1, 7], ["1", "7"], "1,7", "1 7", "[1, 7]" -> list of ints (None if not a label list)."""
    if isinstance(v, list):
        try:
            return [int(x) for x in v]
        except (TypeError, ValueError):
            return None
    if isinstance(v, str):
        toks = re.findall(r"-?\d+", v)
        return [int(t) for t in toks] if toks or v.strip() in ("", "[]") else None
    return None


def read_any_json(p):
    """JSON list, JSON dict wrapping a list / keyed by id, or JSON Lines."""
    raw = open(p, encoding="utf-8").read()
    try:
        d = json.loads(raw)
    except json.JSONDecodeError:
        d = [json.loads(l) for l in raw.splitlines() if l.strip()]
    if isinstance(d, dict):
        lists = [v for v in d.values() if isinstance(v, list) and v and isinstance(v[0], dict)]
        if lists:
            d = lists[0]
        elif all(isinstance(v, dict) for v in d.values()):
            d = [{"id": k, **v} for k, v in d.items()]
    return d


def standardize(d):
    """Rename fields to id / text / labels. Returns (rows, label_key or None, problem or None)."""
    if not (isinstance(d, list) and d and isinstance(d[0], dict)):
        return None, None, f"not a list of records ({type(d).__name__})"
    r0 = d[0]
    ik = next((k for k in ID_KEYS if k in r0), None)
    tk = next((k for k in TEXT_KEYS if k in r0), None)
    if ik is None or tk is None:
        return None, None, f"no id/text field; keys = {list(r0.keys())}"
    lk = next((k for k in LABEL_KEYS if k in r0 and parse_labels(r0[k]) is not None
               and not isinstance(r0[k], (int, bool))), None)
    rows = []
    for r in d:
        row = {"id": str(r[ik]), "text": r[tk]}
        if lk is not None:
            row["labels"] = parse_labels(r.get(lk)) or []
        rows.append(row)
    return rows, ("labels" if lk else None), None


def load_split(name, need_label):
    """Picks the best-matching <name> under /kaggle/input and explains every file it skips."""
    stem = name.split(".")[0]
    hits = [p for ext in (".json", ".jsonl") for p in find_files(stem + ext)]
    hits.sort(key=lambda p: (("lsi" not in p.lower()) and ("phase_5" not in p.lower()) and ("phase5" not in p.lower()), p))
    if not hits:
        print(f"  {name}: no file called {stem}.json / {stem}.jsonl under {INPUT_ROOT}")
    for p in hits:
        try:
            d = read_any_json(p)
        except Exception as e:
            print(f"  skip {p}: unreadable ({e.__class__.__name__}: {str(e)[:80]})"); continue
        if isinstance(d, list) and d and isinstance(d[0], dict) and isinstance(d[0].get("label"), (int, bool)) \
                and not isinstance(d[0].get("labels"), list):
            print(f"  skip {p}: single int 'label' -> this is Phase 4 (CJPE) data"); continue
        rows, lk, problem = standardize(d)
        if problem:
            print(f"  skip {p}: {problem}"); continue
        if need_label and lk is None:
            print(f"  skip {p}: no statute-label list; keys = {list(d[0].keys())}"); continue
        print(f"  {name}: {p} ({len(rows)} rows, labels: {'yes' if lk else 'no'})")
        return rows, lk
    return None, None


def show_inputs():
    print(f"\nFiles under {INPUT_ROOT}:")
    for root, _, files in os.walk(INPUT_ROOT, followlinks=True):
        for f in files:
            if f.endswith((".json", ".jsonl", ".bin", ".pt", ".py")):
                p = os.path.join(root, f)
                print(f"  {os.path.getsize(p) / 2**20:8.1f} MB  {p}")


MODEL_BIN = find_files("pytorch_model.bin")[0]
MODEL_DIR = os.path.dirname(MODEL_BIN)
print("model dir:", MODEL_DIR)
print("data:")
train_raw, LF_TR = load_split("train.json", need_label=True)
dev_raw, LF_DV = load_split("dev.json", need_label=True)
test_raw, _ = load_split("test.json", need_label=False)
if dev_raw is None or test_raw is None:
    show_inputs()
    raise FileNotFoundError("Phase 5 dev/test data not found - see the skip reasons and file list above. "
                            "Attach the Phase_5_LSI dataset via 'Add Input'.")

In [ ]:
# Phase 1 model code + tokenizer
shutil.copy(os.path.join(MODEL_DIR, "model.py"), os.path.join(OUT, "legal_lm_model.py"))
sys.path.insert(0, OUT)
from legal_lm_model import LegalLM, ModelConfig  # noqa: E402
from tokenizers import Tokenizer  # noqa: E402

LM_CFG = json.load(open(os.path.join(MODEL_DIR, "config.json")))
BASE_SD = torch.load(MODEL_BIN, map_location="cpu")
tokenizer = Tokenizer.from_file(os.path.join(MODEL_DIR, "tokenizer.json"))
PAD_ID = 0
print("LM config:", LM_CFG)

In [ ]:
# Data: dedupe, label space, split, tokenize
def text_of(r):
    t = r["text"]
    return " ".join(t) if isinstance(t, list) else str(t)


def labels_of(r, lf):
    return sorted({int(x) for x in r[lf]})


def dedupe(rows, tag):
    seen, out = set(), []
    for r in rows:
        if r["id"] not in seen:
            seen.add(r["id"]); out.append(r)
    if len(out) != len(rows):
        print(f"  {tag}: removed {len(rows) - len(out)} duplicate IDs -> {len(out)}")
    return out


dev_rows = dedupe(dev_raw, "dev")
for r in dev_rows:
    r["_y"] = labels_of(r, LF_DV)
test_rows = dedupe(test_raw, "test")
assert len(test_rows) == len(test_raw), "test.json has duplicate IDs"
for r in test_rows:                      # never touch test labels
    for k in ("labels", "label", "statutes", "statute_ids"):
        r.pop(k, None)

if train_raw is not None:
    train_rows = dedupe(train_raw, "train")
    for r in train_rows:
        r["_y"] = labels_of(r, LF_TR)
    dev_ids = {r["id"] for r in dev_rows}
    n0 = len(train_rows)
    train_rows = [r for r in train_rows if r["id"] not in dev_ids]
    if n0 != len(train_rows):
        print(f"  removed {n0 - len(train_rows)} train docs that also appear in dev")
    val_rows, MODE = dev_rows, "train->dev"
else:
    print("\n" + "!" * 80 + "\n  train.json NOT FOUND - training on 80% of dev, validating on 20%.\n" + "!" * 80)
    rng = random.Random(CFG["seed"]); rows = dev_rows[:]; rng.shuffle(rows)
    k = int(round(len(rows) * CFG["holdout_frac"]))
    val_rows, train_rows, MODE = rows[:k], rows[k:], "dev-split"

if SMOKE:
    rng = random.Random(0)
    train_rows = rng.sample(train_rows, min(40, len(train_rows)))
    val_rows = rng.sample(val_rows, min(16, len(val_rows))); test_rows = test_rows[:10]

# valid statute IDs = every ID seen in the labelled data (the evaluator derives its label set from references)
LABELS = sorted({l for r in train_rows + val_rows for l in r["_y"]})
LID = {l: i for i, l in enumerate(LABELS)}
L = len(LABELS)
print(f"MODE={MODE} | train {len(train_rows)} | val {len(val_rows)} | test {len(test_rows)} | statutes {L} "
      f"(IDs {LABELS[0]}..{LABELS[-1]})")


def normalize(t):
    return re.sub(r"\s+", " ", t).strip()


def multi_hot(ys):
    Y = np.zeros((len(ys), L), dtype=np.float32)
    for i, y in enumerate(ys):
        for l in y:
            if l in LID:
                Y[i, LID[l]] = 1.0
    return Y


def tokenize_rows(rows, tag):
    t0, docs = time.time(), []
    for i in range(0, len(rows), 500):
        batch = rows[i:i + 500]
        enc = tokenizer.encode_batch([normalize(text_of(r)) for r in batch])
        for r, e in zip(batch, enc):
            ids = np.asarray(e.ids if e.ids else [PAD_ID], dtype=np.uint16)
            docs.append(dict(id=r["id"], y=r.get("_y", []), ids=ids, ntok=len(ids)))
    lens = np.array([d["ntok"] for d in docs])
    print(f"  {tag}: {len(docs)} docs | tokens p50 {int(np.median(lens))} p90 {int(np.percentile(lens, 90))} "
          f"max {lens.max()} | {time.time() - t0:.0f}s")
    return docs


train_docs = tokenize_rows(train_rows, "train")
val_docs = tokenize_rows(val_rows, "val")
test_docs = tokenize_rows(test_rows, "test")
del train_raw, dev_raw, test_raw, train_rows, val_rows, test_rows, dev_rows
gc.collect()

Y_TRAIN = multi_hot([d["y"] for d in train_docs])
Y_VAL = multi_hot([d["y"] for d in val_docs])
for d, row in zip(train_docs, Y_TRAIN):
    d["yv"] = row
for d, row in zip(val_docs, Y_VAL):
    d["yv"] = row

pos = Y_TRAIN.sum(0); neg = len(Y_TRAIN) - pos
POS_WEIGHT = torch.tensor(np.clip(np.sqrt(neg / np.maximum(pos, 1)), 1.0, CFG["pos_weight_max"]), dtype=torch.float32)
PRIOR = np.clip(pos / len(Y_TRAIN), 1e-4, 0.5)
# frequency buckets by training support (for rare-statute analysis)
order = np.argsort(-pos)
BUCKET = np.empty(L, dtype=object)
for name, sl in (("frequent", order[:L // 3]), ("mid", order[L // 3:2 * L // 3]), ("rare", order[2 * L // 3:])):
    BUCKET[sl] = name
print(f"labels/doc: train {Y_TRAIN.sum(1).mean():.2f} | val {Y_VAL.sum(1).mean():.2f} | "
      f"train support min/median/max {int(pos.min())}/{int(np.median(pos))}/{int(pos.max())} | "
      f"statutes unseen in train: {int((pos == 0).sum())}")

json.dump([{"id": d["id"], "gold": d["y"]} for d in val_docs], open(os.path.join(OUT, "val_truth.json"), "w"))

# label statistics + co-occurrence (for the analysis section)
with open(os.path.join(OUT, "label_stats.csv"), "w", newline="") as f:
    w = csv.writer(f); w.writerow(["statute_id", "train_support", "val_support", "bucket", "pos_weight"])
    for i, l in enumerate(LABELS):
        w.writerow([l, int(pos[i]), int(Y_VAL[:, i].sum()), BUCKET[i], round(float(POS_WEIGHT[i]), 2)])
co = Counter()
for d in train_docs:
    for a in range(len(d["y"])):
        for b in range(a + 1, len(d["y"])):
            co[(d["y"][a], d["y"][b])] += 1
with open(os.path.join(OUT, "label_cooccurrence_train.csv"), "w", newline="") as f:
    w = csv.writer(f); w.writerow(["statute_a", "statute_b", "count"])
    for (a, b), c in co.most_common(200):
        w.writerow([a, b, c])

In [ ]:
# Chunking + batching
def make_chunks(ids, L_, overlap, K, select):
    n = len(ids)
    if n <= L_:
        return [ids]
    stride = L_ - overlap
    starts = list(range(0, n - L_ + 1, stride))
    if starts[-1] != n - L_:
        starts.append(n - L_)
    if len(starts) > K:
        if select == "first":
            starts = starts[:K]
        elif select == "headtail":
            starts = starts[:1] + starts[-(K - 1):] if K > 1 else starts[:1]
        else:  # spread
            idx = np.unique(np.linspace(0, len(starts) - 1, K).round().astype(int))
            starts = [starts[i] for i in idx]
    return [ids[s:s + L_] for s in starts]


def collate(docs, cc, keep="all"):
    """keep: 'all' | 'first' (opening chunk only) | 'last' (closing chunk only) — section ablation."""
    seqs, doc_idx, slot = [], [], []
    for b, d in enumerate(docs):
        ch = make_chunks(d["ids"], cc["chunk_len"], cc["overlap"], cc["max_chunks"], cc["select"])
        if keep == "first":
            ch = ch[:1]
        elif keep == "last":
            ch = ch[-1:]
        for i, c in enumerate(ch):
            seqs.append(c); doc_idx.append(b); slot.append(i)
    T = max(len(c) for c in seqs)
    ids = torch.full((len(seqs), T), PAD_ID, dtype=torch.long)
    lens = torch.empty(len(seqs), dtype=torch.long)
    for j, c in enumerate(seqs):
        ids[j, :len(c)] = torch.from_numpy(c.astype(np.int64)); lens[j] = len(c)
    out = dict(ids=ids, lens=lens, doc_idx=torch.tensor(doc_idx), slot=torch.tensor(slot), n_docs=len(docs))
    if "yv" in docs[0]:
        out["y"] = torch.from_numpy(np.stack([d["yv"] for d in docs]))
    return out


def to_dev(batch):
    return {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}

In [ ]:
# Model: Phase 1 backbone + chunk transformer + label-wise attention head
class LSIModel(nn.Module):
    def __init__(self, lm, n_labels, d_doc=256, max_slots=64, dropout=0.1):
        super().__init__()
        self.lm = lm
        d = lm.cfg.d_model
        self.frozen_below = 0
        self.grad_ckpt = False
        self.prompt = None                                   # set for prompt tuning
        self.chunk_proj = nn.Sequential(nn.Linear(2 * d, d_doc), nn.GELU(), nn.LayerNorm(d_doc))
        self.slot_emb = nn.Embedding(max_slots, d_doc)
        self.ln1 = nn.LayerNorm(d_doc)
        self.mha = nn.MultiheadAttention(d_doc, 4, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_doc)
        self.ff = nn.Sequential(nn.Linear(d_doc, 2 * d_doc), nn.GELU(), nn.Dropout(dropout), nn.Linear(2 * d_doc, d_doc))
        self.ln3 = nn.LayerNorm(d_doc)
        self.label_q = nn.Parameter(torch.randn(n_labels, d_doc) * 0.02)   # one query per statute
        self.label_w = nn.Parameter(torch.randn(n_labels, d_doc) * 0.02)   # one output vector per statute
        self.label_b = nn.Parameter(torch.tensor(np.log(PRIOR / (1 - PRIOR)), dtype=torch.float32))
        self.drop = nn.Dropout(dropout)
        self.max_slots = max_slots

    def head_parameters(self):
        # the soft prompt is trained with the backbone LR, not the head LR
        return [p for n, p in self.named_parameters() if not n.startswith("lm.") and n != "prompt"]

    def encode(self, ids, lens):
        lm = self.lm
        P = 0 if self.prompt is None else self.prompt.shape[0]
        T = ids.shape[1] + P
        cos, sin = lm.rope_cos[:T][None, None], lm.rope_sin[:T][None, None]
        fb = self.frozen_below
        with (torch.no_grad() if fb > 0 else nullcontext()):
            x = lm.tok(ids)
            if P:
                x = torch.cat([self.prompt[None].expand(x.shape[0], -1, -1).to(x.dtype), x], 1)
            for blk in lm.blocks[:fb]:
                x = blk(x, cos, sin)
        if fb > 0:
            x = x.detach()
        for blk in lm.blocks[fb:]:
            if self.grad_ckpt and self.training:
                x = torch.utils.checkpoint.checkpoint(blk, x, cos, sin, use_reentrant=False)
            else:
                x = blk(x, cos, sin)
        x = lm.norm(x).float()
        pos_ = torch.arange(T, device=ids.device)[None]
        mask = ((pos_ >= P) & (pos_ < (lens + P)[:, None])).float()        # real tokens only
        mean = (x * mask[..., None]).sum(1) / lens[:, None].float()
        last = x[torch.arange(x.shape[0], device=ids.device), lens + P - 1]
        return torch.cat([mean, last], -1)

    def forward(self, b):
        v = self.chunk_proj(self.encode(b["ids"], b["lens"]))
        v = v + self.slot_emb(b["slot"].clamp(max=self.max_slots - 1))
        B, S = b["n_docs"], int(b["slot"].max()) + 1
        H = v.new_zeros(B, S, v.shape[-1])
        H[b["doc_idx"], b["slot"]] = v
        mask = torch.zeros(B, S, dtype=torch.bool, device=v.device)
        mask[b["doc_idx"], b["slot"]] = True
        h = self.ln1(H)
        H = H + self.mha(h, h, h, key_padding_mask=~mask, need_weights=False)[0]
        H = self.ln3(H + self.ff(self.ln2(H))).float()                     # [B, S, d]
        scores = torch.einsum("bsd,ld->bls", H, self.label_q.float())       # [B, L, S]
        w = scores.masked_fill(~mask[:, None, :], float("-inf")).softmax(-1)
        doc_l = torch.einsum("bls,bsd->bld", w, H)                          # [B, L, d]
        logits = (self.drop(doc_l) * self.label_w.float()).sum(-1) + self.label_b
        return logits, w

In [ ]:
# Adaptation methods
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r, alpha, dropout=0.05):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t().to(x.dtype)) @ self.B.t().to(x.dtype) * self.scale


class Adapter(nn.Module):
    def __init__(self, d, b):
        super().__init__()
        self.down, self.up = nn.Linear(d, b), nn.Linear(b, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


class WithAdapter(nn.Module):
    def __init__(self, sub, d, b):
        super().__init__()
        self.sub, self.adapter = sub, Adapter(d, b)

    def forward(self, *a, **k):
        return self.adapter(self.sub(*a, **k))


def fresh_lm(layers=None):
    cfg = dict(LM_CFG)
    if layers is not None:
        cfg["n_layer"] = len(layers)
    lm = LegalLM(ModelConfig(**cfg))
    if layers is None:
        lm.load_state_dict(BASE_SD, strict=True)
    else:
        sd = {}
        for k, v in BASE_SD.items():
            if k.startswith("blocks."):
                i = int(k.split(".")[1])
                if i in layers:
                    sd[k.replace(f"blocks.{i}.", f"blocks.{layers.index(i)}.", 1)] = v
            else:
                sd[k] = v
        lm.load_state_dict(sd, strict=True)
    return lm


def freeze(module):
    for p in module.parameters():
        p.requires_grad = False


def build_model(exp):
    m = exp["method"]
    model = LSIModel(fresh_lm(layers=exp["layers"]) if m == "kd" else fresh_lm(), L)
    lm = model.lm
    if m in ("full", "kd"):
        pass
    elif m == "partial":
        k = len(lm.blocks) - exp["train_top"]
        freeze(lm)
        for blk in lm.blocks[k:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in lm.norm.parameters():
            p.requires_grad = True
        model.frozen_below = k
    elif m == "lora":
        freeze(lm)
        for blk in lm.blocks:
            blk.attn.qkv = LoRALinear(blk.attn.qkv, exp["r"], exp["alpha"])
            blk.attn.proj = LoRALinear(blk.attn.proj, exp["r"], exp["alpha"])
            blk.mlp.w12 = LoRALinear(blk.mlp.w12, exp["r"], exp["alpha"])
            blk.mlp.w3 = LoRALinear(blk.mlp.w3, exp["r"], exp["alpha"])
    elif m == "adapter":
        freeze(lm)
        d = lm.cfg.d_model
        for blk in lm.blocks:
            blk.attn = WithAdapter(blk.attn, d, exp["bottleneck"])
            blk.mlp = WithAdapter(blk.mlp, d, exp["bottleneck"])
            for p in list(blk.norm1.parameters()) + list(blk.norm2.parameters()):
                p.requires_grad = True
        for p in lm.norm.parameters():
            p.requires_grad = True
    elif m == "prompt":
        freeze(lm)
        # soft prompt initialised from embeddings of random real tokens (standard prompt-tuning init)
        idx = torch.randint(1000, LM_CFG["vocab_size"], (exp["n_prompt"],))
        model.prompt = nn.Parameter(lm.tok.weight[idx].detach().clone())
    else:
        raise ValueError(m)
    return model.to(device)


def param_stats(model):
    seen, total, train = set(), 0, 0
    for p in model.parameters():
        if id(p) in seen:
            continue
        seen.add(id(p)); total += p.numel(); train += p.numel() if p.requires_grad else 0
    return total, train

In [ ]:
# Metrics + thresholds
def ml_counts(Y, P):
    Y, P = Y.astype(bool), P.astype(bool)
    tp = (Y & P).sum(0); fp = (~Y & P).sum(0); fn = (Y & ~P).sum(0)
    return tp, fp, fn


def per_label_f1(Y, P):
    tp, fp, fn = ml_counts(Y, P)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)


def metrics(Y, P):
    tp, fp, fn = ml_counts(Y, P)
    micro = 2 * tp.sum() / max(1, 2 * tp.sum() + fp.sum() + fn.sum())
    f1l = per_label_f1(Y, P)
    out = dict(macro_f1=float(f1l.mean()), micro_f1=float(micro),
               subset_acc=float((Y.astype(bool) == P.astype(bool)).all(1).mean()),
               precision_micro=float(tp.sum() / max(1, tp.sum() + fp.sum())),
               recall_micro=float(tp.sum() / max(1, tp.sum() + fn.sum())))
    for bk in ("frequent", "mid", "rare"):
        sel = BUCKET == bk
        out[f"macro_f1_{bk}"] = float(f1l[sel].mean()) if sel.any() else None
    return out


GRID = np.round(np.arange(0.02, 0.96, 0.01), 2)


def best_global(Y, prob):
    best = (0.5, -1)
    for t in GRID:
        f = per_label_f1(Y, prob >= t).mean()
        if f > best[1] + 1e-9:
            best = (float(t), f)
    return best


def best_per_label(Y, prob, t_global, k_shrink=5.0):
    """Per-statute threshold maximising that statute's F1, shrunk toward the global one by support."""
    n_pos = Y.sum(0)
    # F1 for every (threshold, label) at once
    Yb = Y.astype(bool)
    f1 = np.zeros((len(GRID), Y.shape[1]))
    for i, t in enumerate(GRID):
        Pb = prob >= t
        tp = (Yb & Pb).sum(0); fp = (~Yb & Pb).sum(0); fn = (Yb & ~Pb).sum(0)
        den = 2 * tp + fp + fn
        f1[i] = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
    t_raw = GRID[f1.argmax(0)]
    return (n_pos * t_raw + k_shrink * t_global) / (n_pos + k_shrink)


def apply_thr(prob, thr, at_least_one):
    P = prob >= thr
    if at_least_one:
        empty = ~P.any(1)
        P[np.where(empty)[0], prob[empty].argmax(1)] = True
    return P


def choose_thresholds(Y, prob, seed=0):
    """Global vs per-label decided by 2-fold cross-fitting on dev; final thresholds fit on all of dev."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(Y)); a, b = idx[: len(Y) // 2], idx[len(Y) // 2:]
    cf = {"global": [], "per_label": []}
    for tr, te in ((a, b), (b, a)):
        tg, _ = best_global(Y[tr], prob[tr])
        cf["global"].append(per_label_f1(Y[te], prob[te] >= tg).mean())
        tl = best_per_label(Y[tr], prob[tr], tg)
        cf["per_label"].append(per_label_f1(Y[te], prob[te] >= tl).mean())
    cf = {k: float(np.mean(v)) for k, v in cf.items()}
    tg, _ = best_global(Y, prob)
    strategy = "per_label" if cf["per_label"] > cf["global"] else "global"
    thr = best_per_label(Y, prob, tg) if strategy == "per_label" else np.full(Y.shape[1], tg)
    f_plain = per_label_f1(Y, apply_thr(prob, thr, False)).mean()
    f_one = per_label_f1(Y, apply_thr(prob, thr, True)).mean()
    return dict(thr=thr, t_global=tg, strategy=strategy, at_least_one=bool(f_one > f_plain), cf=cf)

In [ ]:
# Training / inference
@torch.no_grad()
def predict(model, docs, cc, keep="all"):
    model.eval()
    t0 = time.time()
    order = np.argsort([d["ntok"] for d in docs])
    out = torch.zeros(len(docs), L)
    for i in range(0, len(docs), cc["infer_docs"]):
        idx = order[i:i + cc["infer_docs"]]
        b = to_dev(collate([docs[j] for j in idx], cc, keep))
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            logits, _ = model(b)
        out[torch.as_tensor(idx)] = logits.float().cpu()
    if USE_AMP:
        torch.cuda.synchronize()
    return out.numpy(), 1000 * (time.time() - t0) / max(1, len(docs))


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


rng_eval = np.random.default_rng(CFG["seed"])
EVAL_IDX = np.sort(rng_eval.choice(len(val_docs), size=min(CFG["eval_subset"], len(val_docs)), replace=False))
EVAL_DOCS = [val_docs[i] for i in EVAL_IDX]


def quick_eval(model, cc):
    lg, _ = predict(model, EVAL_DOCS, cc)
    return best_global(Y_VAL[EVAL_IDX], sigmoid(lg))[1]


def train_model(model, exp, cc, docs, kd_logits=None):
    head_ids = {id(p) for p in model.head_parameters()}
    lm_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    head_params = [p for p in model.head_parameters() if p.requires_grad]
    groups = []
    for ps, lr in ((lm_params, exp["lr"]), (head_params, exp["head_lr"])):
        dec = [p for p in ps if p.dim() >= 2 and p is not model.prompt]
        nod = [p for p in ps if p.dim() < 2 or p is model.prompt]
        groups += [dict(params=dec, lr=lr, weight_decay=0.01), dict(params=nod, lr=lr, weight_decay=0.0)]
    groups = [g for g in groups if g["params"]]
    for g in groups:
        g["base_lr"] = g["lr"]
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98), eps=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    pw = POS_WEIGHT.to(device)

    n, mb, acc = len(docs), cc["micro_docs"], cc["accum"]
    spe = math.ceil(n / (mb * acc))
    sched = dict(total=exp["epochs"] * spe)
    warm = max(3, int(0.06 * sched["total"]))
    budget = exp["minutes"] * 60
    eval_at, t_start, step, best, t_measure = set(), time.time(), 0, (-1.0, None, None), None
    kd_T, kd_a = exp.get("T", 2.0), exp.get("alpha", 0.6)

    def lr_mult(s):
        return (s + 1) / warm if s < warm else max(0.05, (sched["total"] - s) / max(1, sched["total"] - warm))

    def snapshot():
        return {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}

    model.train()
    done = False
    for ep in range(exp["epochs"]):
        perm = np.random.permutation(n)
        micro = [perm[i:i + mb] for i in range(0, n, mb)]
        for mi in range(0, len(micro), acc):
            group = micro[mi:mi + acc]
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * lr_mult(step)
            loss_v = 0.0
            for idx in group:
                b = to_dev(collate([docs[j] for j in idx], cc))
                try:
                    with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                        logits, _ = model(b)
                    loss = F.binary_cross_entropy_with_logits(logits, b["y"], pos_weight=pw)
                    if kd_logits is not None:
                        t_log = kd_logits[torch.as_tensor(idx)].to(device)
                        kd = F.binary_cross_entropy_with_logits(logits / kd_T, torch.sigmoid(t_log / kd_T)) * kd_T ** 2
                        loss = kd_a * kd + (1 - kd_a) * loss
                    scaler.scale(loss / len(group)).backward()
                    loss_v += loss.item() / len(group)
                except torch.cuda.OutOfMemoryError:
                    print("  OOM -> enabling gradient checkpointing, skipping this micro-batch")
                    opt.zero_grad(set_to_none=True); torch.cuda.empty_cache(); model.grad_ckpt = True
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            step += 1
            if step == 2:
                t_measure = time.time()
            if step == 7:
                sec_per_step = (time.time() - t_measure) / 5
                fit = step + int(max(0.0, budget - (time.time() - t_start)) / sec_per_step)
                sched["total"] = max(step + 1, min(sched["total"], fit))
                eval_at = {max(step + 1, int(round(sched["total"] * f))) for f in np.linspace(0.5, 1.0, CFG["n_evals"])}
                print(f"  {sec_per_step:.2f}s/step | planned {exp['epochs'] * spe} -> running {sched['total']} steps "
                      f"({sched['total'] * mb * acc / n:.2f} epochs) | evals at {sorted(eval_at)}")
            if step % 25 == 0:
                print(f"  step {step}/{sched['total']} | loss {loss_v:.4f} | lr x{lr_mult(step):.2f} | "
                      f"{(time.time() - t_start) / 60:.1f} min")
            last = step >= sched["total"] or (time.time() - t_start) > budget * 1.15 \
                or elapsed_h() > CFG["total_hours"] - 0.2
            if step in eval_at or last:
                f = quick_eval(model, cc)
                print(f"  [eval step {step}] dev-subset macro-F1 (best global thr) {f:.4f}")
                if f > best[0]:
                    best = (f, step, snapshot())
                model.train()
            if last:
                done = True
                break
        if done:
            break
    if best[2] is None:
        best = (quick_eval(model, cc), step, snapshot())
    model.load_state_dict(best[2])
    print(f"  best dev-subset macro-F1 {best[0]:.4f} at step {best[1]} | train {(time.time() - t_start) / 60:.1f} min")
    return dict(train_min=(time.time() - t_start) / 60, steps=step)

In [ ]:
# Outputs
RESULTS, VAL_PROBS, TEST_PROBS = [], {}, {}


def preds_to_ids(P):
    return [[int(LABELS[j]) for j in np.flatnonzero(row)] for row in P]


def write_submission(name, P_test):
    preds = [{"id": d["id"], "pred": ids} for d, ids in zip(test_docs, preds_to_ids(P_test))]
    valid = set(LABELS)
    assert len(preds) == len(test_docs) and len({p["id"] for p in preds}) == len(preds)
    assert all(len(set(p["pred"])) == len(p["pred"]) and set(p["pred"]) <= valid for p in preds)
    tag = ("smoke_" if SMOKE else "") + name
    jpath = os.path.join(OUT, f"{tag}_prediction.json")
    json.dump(preds, open(jpath, "w"))
    with zipfile.ZipFile(os.path.join(OUT, f"{tag}.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jpath, arcname="prediction.json")
    card = np.mean([len(p["pred"]) for p in preds])
    print(f"  -> {tag}.zip  (avg statutes/test doc {card:.2f}, empty {sum(not p['pred'] for p in preds)})")


def record(name, method, val_prob, test_prob, extra):
    th = choose_thresholds(Y_VAL, val_prob, CFG["seed"])
    P_val = apply_thr(val_prob, th["thr"], th["at_least_one"])
    m_final = metrics(Y_VAL, P_val)
    m_05 = metrics(Y_VAL, val_prob >= 0.5)
    m_g = metrics(Y_VAL, val_prob >= th["t_global"])
    row = dict(name=name, method=method,
               val_macro_f1=round(m_final["macro_f1"], 4), val_micro_f1=round(m_final["micro_f1"], 4),
               val_subset_acc=round(m_final["subset_acc"], 4),
               val_precision_micro=round(m_final["precision_micro"], 4), val_recall_micro=round(m_final["recall_micro"], 4),
               macro_f1_frequent=round(m_final["macro_f1_frequent"] or 0, 4), macro_f1_mid=round(m_final["macro_f1_mid"] or 0, 4),
               macro_f1_rare=round(m_final["macro_f1_rare"] or 0, 4),
               macro_f1_thr05=round(m_05["macro_f1"], 4), macro_f1_global_thr=round(m_g["macro_f1"], 4),
               global_thr=th["t_global"], thr_strategy=th["strategy"], at_least_one=th["at_least_one"],
               cf_global=round(th["cf"]["global"], 4), cf_per_label=round(th["cf"]["per_label"], 4), **extra)
    RESULTS.append(row)
    VAL_PROBS[name], TEST_PROBS[name] = val_prob, test_prob

    json.dump([{"id": d["id"], "pred": ids} for d, ids in zip(val_docs, preds_to_ids(P_val))],
              open(os.path.join(OUT, f"val_pred_{name}.json"), "w"))
    tp, fp, fn = ml_counts(Y_VAL, P_val)
    f1l = per_label_f1(Y_VAL, P_val)
    with open(os.path.join(OUT, f"per_label_{name}.csv"), "w", newline="") as f:
        w = csv.writer(f); w.writerow(["statute_id", "bucket", "train_support", "val_support", "threshold",
                                       "tp", "fp", "fn", "precision", "recall", "f1"])
        for i, l in enumerate(LABELS):
            w.writerow([l, BUCKET[i], int(pos[i]), int(Y_VAL[:, i].sum()), round(float(th["thr"][i]), 3),
                        int(tp[i]), int(fp[i]), int(fn[i]), round(tp[i] / max(1, tp[i] + fp[i]), 4),
                        round(tp[i] / max(1, tp[i] + fn[i]), 4), round(float(f1l[i]), 4)])
    # confusions: a gold statute missed while a wrong statute was predicted on the same document
    conf = Counter()
    Yb, Pb = Y_VAL.astype(bool), P_val.astype(bool)
    for r in range(len(Yb)):
        missed, wrong = np.flatnonzero(Yb[r] & ~Pb[r]), np.flatnonzero(~Yb[r] & Pb[r])
        for a in missed:
            for c in wrong:
                conf[(LABELS[a], LABELS[c])] += 1
    with open(os.path.join(OUT, f"confusions_{name}.csv"), "w", newline="") as f:
        w = csv.writer(f); w.writerow(["missed_gold_statute", "predicted_instead", "docs"])
        for (a, c), k in conf.most_common(50):
            w.writerow([a, c, k])
    write_submission(name, apply_thr(test_prob, th["thr"], th["at_least_one"]))
    with open(os.path.join(OUT, "results.csv"), "w", newline="") as f:
        keys = list(dict.fromkeys(k for r in RESULTS for k in r))
        w = csv.DictWriter(f, fieldnames=keys); w.writeheader(); w.writerows(RESULTS)
    print("  ", row)


def exp_cc(exp):
    return {**CFG, **{k: exp[k] for k in ("chunk_len", "overlap", "max_chunks", "select", "micro_docs",
                                          "accum", "infer_docs") if k in exp}}


def free(*objs):
    gc.collect()
    if USE_AMP:
        torch.cuda.empty_cache()


def run_experiment(exp, kd_logits=None, train_subset=None):
    print("\n" + "=" * 90 + f"\n{exp['name']}  ({exp['method']})  | notebook time {elapsed_h():.2f} h\n" + "=" * 90)
    seed_all(CFG["seed"])
    cc = exp_cc(exp)
    model = build_model(exp)
    total, trainable = param_stats(model)
    print(f"  params: total {total / 1e6:.2f}M | trainable {trainable / 1e6:.3f}M ({100 * trainable / total:.2f}%)")
    if USE_AMP:
        torch.cuda.reset_peak_memory_stats()
    docs = train_docs if train_subset is None else [train_docs[i] for i in train_subset]
    info = train_model(model, exp, cc, docs, kd_logits)
    peak = torch.cuda.max_memory_allocated() / 2 ** 30 if USE_AMP else 0.0
    val_lg, ms_val = predict(model, val_docs, cc)
    test_lg, ms_test = predict(model, test_docs, cc)
    extra = dict(total_params_M=round(total / 1e6, 2), trainable_params_M=round(trainable / 1e6, 3),
                 trainable_pct=round(100 * trainable / total, 2), size_fp16_MB=round(total * 2 / 2 ** 20, 1),
                 peak_train_mem_GB=round(peak, 2), train_min=round(info["train_min"], 1), steps=info["steps"],
                 infer_ms_per_doc=round((ms_val + ms_test) / 2, 1),
                 chunks=f"{cc['max_chunks']}x{cc['chunk_len']} {cc['select']}")
    record(exp["name"], exp["method"], sigmoid(val_lg), sigmoid(test_lg), extra)
    return model, cc

In [ ]:
# 01 — Full fine-tuning (teacher) + section ablation + teacher logits for KD
E = {e["name"]: e for e in EXPS}
teacher, tcc = run_experiment(E["01_fullft_lwan"])

ab = []
for keep in ("all", "first", "last"):
    lg, _ = predict(teacher, val_docs, tcc, keep=keep)
    pr = sigmoid(lg)
    tg, f = best_global(Y_VAL, pr)
    m = metrics(Y_VAL, pr >= tg)
    ab.append(dict(input=keep, macro_f1_global_thr=round(f, 4), micro_f1=round(m["micro_f1"], 4),
                   macro_f1_rare=round(m["macro_f1_rare"] or 0, 4)))
    print("  section ablation:", ab[-1])
with open(os.path.join(OUT, "section_ablation_teacher.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(ab[0])); w.writeheader(); w.writerows(ab)

rng = np.random.default_rng(CFG["seed"])
KD_SUBSET = np.sort(rng.choice(len(train_docs), size=min(CFG["kd_pool"], len(train_docs)), replace=False))
t0 = time.time()
kd_np, _ = predict(teacher, [train_docs[i] for i in KD_SUBSET], tcc)
KD_LOGITS = torch.tensor(kd_np, dtype=torch.float32)
print(f"  teacher logits for {len(KD_SUBSET)} train docs in {(time.time() - t0) / 60:.1f} min")
del teacher; free()

In [ ]:
# 02 — LoRA
m, _ = run_experiment(E["02_lora_lwan"]); del m; free()

In [ ]:
# 03 — Adapters
m, _ = run_experiment(E["03_adapter_lwan"]); del m; free()

In [ ]:
# 04 — Knowledge distillation (12-layer full-FT teacher -> 6-layer student from Phase 1 layers)
m, _ = run_experiment(E["04_kd_student6"], kd_logits=KD_LOGITS, train_subset=KD_SUBSET); del m; free()

In [ ]:
# 05 — Partial / layer-wise fine-tuning (top 4 blocks)
m, _ = run_experiment(E["05_partial_top4"]); del m; free()

In [ ]:
# 06 — Prompt tuning (soft prompt prepended to every chunk; backbone frozen)
m, _ = run_experiment(E["06_prompt_tuning"]); del m; free()

In [ ]:
# 07 — Ensemble (average probabilities of all models so far)
names = list(VAL_PROBS)
print("\n07_ensemble_avg over:", names)
record("07_ensemble_avg", "ensemble(" + "+".join(n[:2] for n in names) + ")",
       np.mean([VAL_PROBS[n] for n in names], 0), np.mean([TEST_PROBS[n] for n in names], 0), dict(steps=0))

In [ ]:
# 08 — Flat baseline: LoRA on the first 1024 tokens only (skipped if out of time)
exp8 = E["08_lora_first1024"]
if elapsed_h() + (exp8["minutes"] + 10) / 60 < CFG["total_hours"]:
    m, _ = run_experiment(exp8); del m; free()
else:
    print(f"Skipping 08 — {elapsed_h():.2f} h used")

In [ ]:
# Summary
print(f"\nTotal notebook time: {elapsed_h():.2f} h | MODE={MODE} | statutes={L}")
cols = ["name", "val_macro_f1", "val_micro_f1", "val_subset_acc", "macro_f1_rare", "macro_f1_thr05",
        "thr_strategy", "at_least_one", "trainable_params_M", "peak_train_mem_GB", "train_min", "infer_ms_per_doc"]
print(" | ".join(cols))
for r in RESULTS:
    print(" | ".join(str(r.get(c, "")) for c in cols))
print("\nSubmission zips:", sorted(f for f in os.listdir(OUT) if f.endswith(".zip")))

In [ ]:
# Download links — one bundle with every submission + clickable links (works in interactive sessions)
from IPython.display import FileLink, FileLinks, display, HTML  # noqa: E402

subs = sorted(f for f in os.listdir(OUT) if f.endswith(".zip") and not f.startswith("ALL_"))
bundle = os.path.join(OUT, "ALL_submissions_and_reports.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in subs:
        z.write(os.path.join(OUT, f), arcname=f"submissions/{f}")
    for f in sorted(os.listdir(OUT)):
        if f.endswith(".csv") or f.startswith("val_"):
            z.write(os.path.join(OUT, f), arcname=f"reports/{f}")
print(f"Bundle: {bundle} ({os.path.getsize(bundle) / 2**20:.1f} MB) with {len(subs)} submissions")
os.chdir(OUT)                       # FileLink paths must be relative to the notebook's working dir
display(HTML("<b>Click to download:</b>"))
display(FileLink(os.path.basename(bundle)))
for f in subs:
    display(FileLink(f))